# argentina.clean — Pruebas interactivas

Recorrido paso a paso del módulo `argentina.clean`.

Funciones simples para limpieza de texto y columnas. Solo stdlib (pandas se usa de forma diferida en `limpiar_columnas` y `porcentaje_nulls`).

## 1. Setup e imports

In [1]:
import argentina as arg

print(f"argentina v{arg.__version__}")

argentina v0.0.14


## 2. quitar_tildes

Saca tildes y marcas diacríticas, preservando mayúsculas/minúsculas y el resto del texto.

In [2]:
arg.clean.quitar_tildes("Córdoba")

'Cordoba'

In [3]:
arg.clean.quitar_tildes("Tucumán, Río Negro, Neuquén")

'Tucuman, Rio Negro, Neuquen'

In [4]:
# La ñ es un caso especial: NFKD la descompone
arg.clean.quitar_tildes("años")

'anos'

In [5]:
# None pasa transparente
print(arg.clean.quitar_tildes(None))

None


In [6]:
# Acepta no-str y lo convierte
arg.clean.quitar_tildes(42)

'42'

## 3. normalizar_texto

Pipeline: quitar tildes → minúsculas → colapsar espacios → strip.

In [7]:
arg.clean.normalizar_texto("  Código   de Provincia ")

'codigo de provincia'

In [8]:
arg.clean.normalizar_texto("\tBUENOS\n\nAIRES ")

'buenos aires'

In [9]:
print(arg.clean.normalizar_texto(None))

None


## 4. snake_case

Útil para normalizar nombres de columnas o identificadores. Reemplaza cualquier cosa que no sea `[a-z0-9]` por `_`, colapsa repetidos y limpia los bordes.

In [10]:
arg.clean.snake_case("Código de Provincia")

'codigo_de_provincia'

In [11]:
arg.clean.snake_case("Edad (años)")

'edad_anos'

In [12]:
# Signos consecutivos colapsan a un solo _
arg.clean.snake_case("  --Hola!!  Mundo??  ")

'hola_mundo'

In [13]:
print(arg.clean.snake_case(None))

None


## 5. limpiar_columnas

Aplica `snake_case` a todas las columnas de un `DataFrame`. Devuelve una copia (no muta el original).

In [14]:
import pandas as pd

df = pd.DataFrame({
    "Código de Provincia": [6, 14, 82],
    "Nombre Provincia": ["Buenos Aires", "Córdoba", "Santa Fe"],
    "Población (2022)": [17_523_996, 3_840_905, 3_536_418],
})
df

,Código de Provincia,Nombre Provincia,Población (2022)
0,6,Buenos Aires,17523996
1,14,Córdoba,3840905
2,82,Santa Fe,3536418


In [15]:
limpio = arg.clean.limpiar_columnas(df)
limpio

,codigo_de_provincia,nombre_provincia,poblacion_2022
0,6,Buenos Aires,17523996
1,14,Córdoba,3840905
2,82,Santa Fe,3536418


In [16]:
# El original queda intacto
list(df.columns)

['Código de Provincia', 'Nombre Provincia', 'Población (2022)']

## 6. porcentaje_nulls

Devuelve un dict `{columna: porcentaje_de_nulos}` (0–100).

In [17]:
df = pd.DataFrame({
    "completa": [1, 2, 3, 4],
    "mitad": [1, None, 3, None],
    "vacia": [None, None, None, None],
})
arg.clean.porcentaje_nulls(df)

{'completa': 0.0, 'mitad': 50.0, 'vacia': 100.0}

In [18]:
# DataFrame vacío → 0.0 en cada columna (no rompe por división por cero)
vacio = pd.DataFrame({"a": [], "b": []})
arg.clean.porcentaje_nulls(vacio)

{'a': 0.0, 'b': 0.0}

## 7. Combinando con el resto del paquete

Ejemplo: armar un DataFrame con provincias y limpiar nombres.

In [19]:
from argentina import provincias

df = pd.DataFrame([
    {"Nombre": p.nombre, "Código INDEC": p.codigo_indec, "Región": p.region}
    for p in provincias.listar()
])
df.head()

,Nombre,Código INDEC,Región
0,Ciudad Autónoma de Buenos Aires,02,CABA
1,Buenos Aires,06,Pampeana
2,Catamarca,10,NOA
3,Córdoba,14,Pampeana
4,Corrientes,18,NEA


In [20]:
limpio = arg.clean.limpiar_columnas(df)
limpio.head()

,nombre,codigo_indec,region
0,Ciudad Autónoma de Buenos Aires,02,CABA
1,Buenos Aires,06,Pampeana
2,Catamarca,10,NOA
3,Córdoba,14,Pampeana
4,Corrientes,18,NEA


In [21]:
# Normalizar valores para hacer join/match sin sufrir las tildes
limpio["nombre_normalizado"] = limpio["nombre"].map(arg.clean.normalizar_texto)
limpio[["nombre", "nombre_normalizado"]].head()

,nombre,nombre_normalizado
0,Ciudad Autónoma de Buenos Aires,ciudad autonoma de buenos aires
1,Buenos Aires,buenos aires
2,Catamarca,catamarca
3,Córdoba,cordoba
4,Corrientes,corrientes


## 8. Tests automáticos

Los tests viven en `tests/test_clean.py`. Para correrlos:

```bash
cd /Users/tobiasyatche/argentina
pytest tests/test_clean.py -v
```

## Notas sueltas / TODOs

- Módulo sin dependencias obligatorias: `quitar_tildes`, `normalizar_texto` y `snake_case` corren con stdlib pura.
- `limpiar_columnas` y `porcentaje_nulls` esperan un `DataFrame` — pandas se asume instalado solo si las llamás.
- `snake_case` normaliza con NFKD: la `ñ` se descompone (`años` → `anos`). Si en algún momento se quiere preservar, hay que tratarla aparte antes del NFKD.